# Klasifikasi Ganjil-Genap Kendaraan dari Plat Nomor dengan Pengolahan Citra Digital

**Kelompok 7 Pengolahan Citra Digital**
Anggota:
- Naura Alifa Damayanti
- Andi Nabil Safaraz
- Muhammad Iqbal Hilmi
- Dania Hafiza  
- Jundan Saiful Haq

Sistem ini menerima foto jalan raya, mencari semua plat nomor di dalamnya, lalu memberi label **Ganjil** atau **Genap** untuk setiap plat dari digit terakhirnya. Semua tahap memakai pengolahan citra konvensional, tanpa deep learning:

1. **Akuisisi dan enhancement:** grayscale, CLAHE (perluasan histogram equalization), median filter.
2. **Lokalisasi ROI plat:** Sobel-X dan Canny, threshold Otsu, closing/opening, filter bentuk kontur, verifikasi kandidat, lalu non-maximum suppression. Satu citra bisa menghasilkan beberapa ROI.
3. **Segmentasi karakter:** Otsu, opening/closing, connected component labeling, lalu pemilihan digit terakhir.
4. **Representasi fitur:** HOG (berbasis tepi), dengan opsi tambahan LBP (tekstur).
5. **Klasifikasi:** KNN, SVM, dan XGBoost untuk digit 0–9, lalu `digit % 2` untuk ganjil/genap.

**Cara menjalankan:** di Kaggle, tambahkan dua dataset (*Indonesian License Plate Dataset* dan *Indonesian License Plate Recognition Dataset*) lewat *Add Input*, lalu *Run All*. Di Colab, ekstrak kedua dataset ke `/content/`. Path dicari otomatis.

## 0. Setup

In [ ]:
import os
import re
import glob
import time
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from skimage.feature import hog, local_binary_pattern
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 5)
sns.set_style("whitegrid")
print("OpenCV:", cv2.__version__)

Semua parameter ada di sini. Nilai awal lokalisasi diambil dari notebook versi pertama; `crop_top_ratio`, metode enhancement, dan `min_plate_score` dipilih ulang di bagian 3 memakai split val.

In [ ]:
CFG = {
    "seed": 42,
    # --- Enhancement ---
    "enhance_method": "clahe",        # "clahe" | "he" | "none"
    "clahe_clip": 2.0,
    "clahe_grid": (8, 8),
    "median_ksize": 3,
    # --- Lokalisasi: pembangkit kandidat ---
    "crop_top_ratio": 0.50,           # bagian atas frame dibuang (langit, gedung)
    "sobel_close_kernel": (17, 3),
    "canny_close_kernel": (15, 3),
    "open_kernel": (3, 3),
    "plate_ar_range": (2.2, 4.5),     # rasio lebar/tinggi plat
    "plate_area_range": (0.0004, 0.025),  # luas kandidat relatif terhadap luas citra
    "plate_min_extent": 0.40,
    "max_candidates": 30,
    # --- Lokalisasi: verifikasi kandidat ---
    "max_mean_saturation": 85.0,
    "min_contrast_gap": 50.0,
    "fg_ratio_range": (0.10, 0.65),
    "n_chars_range": (4, 9),
    "max_h_cv": 0.30,
    "max_y_spread": 0.14,
    # --- Lokalisasi: banyak ROI per citra ---
    "min_plate_score": 0.10,          # ROI dengan skor di bawah ini dibuang (dipilih ulang di val)
    "nms_iou": 0.30,                  # ROI yang tumpang tindih lebih dari ini dengan ROI berskor lebih tinggi dibuang
    "max_plates": 6,                  # batas jumlah ROI per citra
    # --- Segmentasi karakter ---
    "roi_height": 80,
    "use_morphology": True,
    "digit_size": (32, 32),
    # --- Fitur ---
    "use_lbp": False,
    # --- Evaluasi ---
    "iou_match": 0.30,                # ROI dianggap mengenai plat GT jika IoU >= ini
    "iou_digit": 0.50,                # digit terpilih dianggap benar jika IoU dengan bbox digit GT >= ini
}
np.random.seed(CFG["seed"])

In [ ]:
SEARCH_ROOTS = ["/kaggle/input", "/content", "."]


def find_dataset_root(marker: str):
    """Mencari folder dataset berdasarkan nama file/folder penanda di dalamnya."""
    for root in SEARCH_ROOTS:
        if not os.path.exists(root):
            continue
        hits = glob.glob(os.path.join(root, "**", marker), recursive=True)
        if hits:
            return str(Path(sorted(hits, key=len)[0]).parent)
    return None


MAIN_DIR = find_dataset_root("labelswithLP")      # Indonesian License Plate Dataset
REC_DIR = find_dataset_root("classes.names")      # Indonesian License Plate Recognition Dataset
assert MAIN_DIR is not None, "Indonesian License Plate Dataset tidak ditemukan. Tambahkan lewat Add Input."
assert REC_DIR is not None, "Indonesian License Plate Recognition Dataset tidak ditemukan. Tambahkan lewat Add Input."

OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
SPLITS = ["train", "val", "test"]
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

print("MAIN_DIR:", MAIN_DIR)
print("REC_DIR :", REC_DIR)
print("OUT_DIR :", OUT_DIR)

## 1. Ground truth per plat (Indonesian License Plate Dataset)

Setiap baris di `labelswithLP` berisi satu plat: `kelas cx cy w h TEKS`. Satu gambar bisa berisi beberapa baris. Tabel `plates` menyimpan satu baris per plat, termasuk bbox. Bbox ini tidak masuk ke pipeline, melainkan hanya dipakai untuk mencocokkan hasil lokalisasi dengan plat yang benar dan untuk evaluasi.

In [ ]:
def read_yolo_rows(path: str):
    rows = []
    if not os.path.exists(path):
        return rows
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.split()
            if len(parts) >= 5:
                rows.append(parts)
    return rows


def last_digit_of_plate(text: str):
    """Digit terakhir dari kelompok angka plat Indonesia, misal B1184UJY -> 4."""
    cleaned = re.sub(r"[^A-Z0-9]", "", str(text).upper())
    match = re.match(r"^[A-Z]{1,2}(\d{1,4})[A-Z]{0,3}$", cleaned)
    if match:
        return int(match.group(1)[-1])
    digits = re.findall(r"\d", cleaned)
    return int(digits[-1]) if digits else None


def load_plates(split: str) -> pd.DataFrame:
    img_dir = os.path.join(MAIN_DIR, "images", split)
    lbl_dir = os.path.join(MAIN_DIR, "labelswithLP", split)
    records = []
    for img_name in sorted(f for f in os.listdir(img_dir) if f.lower().endswith(IMG_EXT)):
        stem = os.path.splitext(img_name)[0]
        rows = read_yolo_rows(os.path.join(lbl_dir, stem + ".txt"))
        for k, parts in enumerate(rows, start=1):
            text = "".join(parts[5:]).upper()
            digit = last_digit_of_plate(text) if text else None
            records.append({
                "split": split,
                "image_name": img_name,
                "image_path": os.path.join(img_dir, img_name),
                "stem": stem,
                "plate_idx": k,
                "cx": float(parts[1]), "cy": float(parts[2]),
                "bw": float(parts[3]), "bh": float(parts[4]),
                "plate_text": text,
                "last_digit": digit,
                "parity": None if digit is None else digit % 2,
            })
    return pd.DataFrame(records)


plates = pd.concat([load_plates(s) for s in SPLITS], ignore_index=True)
images = (
    plates.groupby(["split", "image_name", "image_path"], as_index=False)
    .agg(n_plates=("plate_idx", "size"))
)

print("Jumlah plat per split :", plates.groupby("split").size().to_dict())
print("Jumlah gambar per split:", images.groupby("split").size().to_dict())
print("Plat tanpa digit terbaca:", int(plates["last_digit"].isna().sum()))
print("\nJumlah plat per gambar (berapa gambar):")
display(images.pivot_table(index="n_plates", columns="split", values="image_name", aggfunc="count").fillna(0).astype(int))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
train_plates = plates[(plates["split"] == "train") & plates["last_digit"].notna()]
sns.countplot(x=train_plates["last_digit"].astype(int), color="#4C72B0", ax=axes[0])
axes[0].set_title("Distribusi digit terakhir (plat train)")
axes[0].set_xlabel("Digit terakhir")
sns.countplot(x=images[images["split"] == "train"]["n_plates"], color="#55A868", ax=axes[1])
axes[1].set_title("Jumlah plat per gambar (train)")
axes[1].set_xlabel("Plat dalam satu gambar")
plt.tight_layout()
plt.show()

## 2. Data karakter (Indonesian License Plate Recognition Dataset)

Dataset ini berisi crop plat (`train001_1.jpg`, `train002_1.jpg`, `train002_2.jpg`, ...) dengan bbox per karakter dan kelas `0–9, A–Z`. Nama file menunjukkan crop ke-k dari gambar di dataset utama, dan pembagian train/val/test-nya sama. Karena itu, classifier digit dilatih dari crop train saja, sehingga gambar test tidak pernah terlihat saat pelatihan.

Sel berikut juga mengecek apakah teks hasil menyusun karakter dari kiri ke kanan sama dengan teks plat di dataset utama.

In [ ]:
with open(os.path.join(REC_DIR, "classes.names"), "r", encoding="utf-8") as f:
    CLASSES = [ln.strip() for ln in f if ln.strip()]
print("Kelas:", "".join(CLASSES))


def load_recognition(split: str) -> pd.DataFrame:
    img_dir = os.path.join(REC_DIR, "images", split)
    lbl_dir = os.path.join(REC_DIR, "labels", split)
    records = []
    for img_name in sorted(f for f in os.listdir(img_dir) if f.lower().endswith(IMG_EXT)):
        stem = os.path.splitext(img_name)[0]
        m = re.match(r"^(.*)_(\d+)$", stem)
        src_stem, plate_idx = (m.group(1), int(m.group(2))) if m else (stem, 1)
        chars = []
        for parts in read_yolo_rows(os.path.join(lbl_dir, stem + ".txt")):
            cls = int(parts[0])
            chars.append((cls, float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])))
        chars.sort(key=lambda c: c[1])  # kiri ke kanan
        records.append({
            "split": split,
            "crop_path": os.path.join(img_dir, img_name),
            "stem": src_stem,
            "plate_idx": plate_idx,
            "rec_text": "".join(CLASSES[c[0]] for c in chars),
            "chars": chars,
        })
    return pd.DataFrame(records)


rec = pd.concat([load_recognition(s) for s in SPLITS], ignore_index=True)
print("Jumlah crop per split:", rec.groupby("split").size().to_dict())

# Cek kecocokan dengan dataset utama
check = rec.merge(plates[["split", "stem", "plate_idx", "plate_text"]], on=["split", "stem", "plate_idx"], how="left")
same_idx = (check["rec_text"] == check["plate_text"]).mean()
texts_per_image = plates.groupby(["split", "stem"])["plate_text"].apply(set).to_dict()
same_image = np.mean([
    r["rec_text"] in texts_per_image.get((r["split"], r["stem"]), set()) for _, r in rec.iterrows()
])
print(f"Teks crop == teks plat dengan indeks sama : {same_idx:.1%}")
print(f"Teks crop ada di gambar asal yang sama     : {same_image:.1%}")

## 3. Enhancement dan lokalisasi ROI plat

Di bagian ini kita nyari dulu posisi plat nomor di dalam frame. Pertama, bagian atas frame dibuang (`crop_top_ratio`), terus sisanya diubah ke grayscale, dikasih CLAHE atau HE biar kontrasnya naik, lalu dihaluskan pakai median filter buat ngurangin noise. Setelah itu prosesnya dibagi jadi dua cabang. Cabang pertama pakai Sobel-X buat ngambil tepi vertikal, soalnya goresan karakter plat banyak yang vertikal. Hasilnya di-Otsu, lalu dikasih closing horizontal dan opening. Cabang kedua pakai Canny, dengan threshold yang diambil dari median intensitas citra, dan hasilnya juga dikasih closing dan opening.

Kontur dari kedua cabang itu kemudian disaring berdasarkan rasio lebar/tinggi, luas, dan *extent* (semacam boundary descriptor sederhana). Dari situ, kandidat terbesar sebanyak `max_candidates` dinilai pakai `score_plate_candidate`. Penilaiannya lihat apakah saturasi HSV-nya rendah, kontras Otsu-nya cukup, dan ada sekitar 4 sampai 9 komponen mirip karakter yang tingginya seragam dan posisinya sejajar. Semua kandidat yang skornya minimal `min_plate_score` dipertahankan. Kalau ada kotak yang numpuk dengan kotak berskor lebih tinggi (IoU lebih dari `nms_iou`), kotak itu dibuang pakai *non-maximum suppression*. Jadi hasil akhirnya bisa lebih dari satu ROI per citra.

In [ ]:
def enhance(gray_or_bgr: np.ndarray, cfg=CFG):
    """Grayscale -> CLAHE / histogram equalization -> median filter."""
    gray = cv2.cvtColor(gray_or_bgr, cv2.COLOR_BGR2GRAY) if gray_or_bgr.ndim == 3 else gray_or_bgr
    if cfg["enhance_method"] == "clahe":
        enhanced = cv2.createCLAHE(clipLimit=cfg["clahe_clip"], tileGridSize=cfg["clahe_grid"]).apply(gray)
    elif cfg["enhance_method"] == "he":
        enhanced = cv2.equalizeHist(gray)
    else:
        enhanced = gray.copy()
    denoised = cv2.medianBlur(enhanced, cfg["median_ksize"])
    return gray, enhanced, denoised


def score_plate_candidate(roi_bgr: np.ndarray, cfg=CFG) -> float:
    """Skor kandidat plat; 0 berarti ditolak."""
    if roi_bgr is None or roi_bgr.size == 0:
        return 0.0
    h, w = roi_bgr.shape[:2]
    if h < 10 or w < 25:
        return 0.0

    # Saturasi rendah: plat Indonesia hitam-putih (Topik 03)
    mean_sat = float(np.mean(cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)[:, :, 1]))
    if mean_sat > cfg["max_mean_saturation"]:
        return 0.0

    # Otsu pada ROI yang tingginya distandarkan (Topik 07)
    target_h = 60
    new_w = max(20, int(round(w * target_h / float(h))))
    gray = cv2.cvtColor(cv2.resize(roi_bgr, (new_w, target_h), interpolation=cv2.INTER_CUBIC), cv2.COLOR_BGR2GRAY)
    blur = cv2.medianBlur(gray, 3)
    _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    fg, bg = gray[binary == 255], gray[binary == 0]
    if len(fg) < 20 or len(bg) < 20:
        return 0.0
    contrast_gap = abs(float(fg.mean()) - float(bg.mean()))
    if contrast_gap < cfg["min_contrast_gap"]:
        return 0.0

    # Polaritas: karakter selalu putih
    border = np.concatenate([binary[0, :], binary[-1, :], binary[:, 0], binary[:, -1]])
    if border.mean() > 127:
        binary = cv2.bitwise_not(binary)
    fg_ratio = float(np.mean(binary == 255))
    lo, hi = cfg["fg_ratio_range"]
    if not (lo <= fg_ratio <= hi):
        return 0.0

    # Komponen mirip karakter (CCL, Topik 07)
    n, _, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    chars = []
    for i in range(1, n):
        cx, cy, cw, ch, area = stats[i]
        ar = cw / float(ch) if ch > 0 else 0
        if (0.28 <= ch / target_h <= 0.85) and (0.02 <= cw / new_w <= 0.22) and (0.14 <= ar <= 1.05) and area >= 20:
            chars.append((cx, cy + ch / 2.0, ch))
    n_lo, n_hi = cfg["n_chars_range"]
    if not (n_lo <= len(chars) <= n_hi):
        return 0.0

    chars.sort(key=lambda c: c[0])
    heights = np.array([c[2] for c in chars], dtype=np.float32)
    y_centers = np.array([c[1] for c in chars], dtype=np.float32)
    h_cv = float(heights.std() / (heights.mean() + 1e-6))
    y_spread = float(y_centers.std() / target_h)
    if h_cv > cfg["max_h_cv"] or y_spread > cfg["max_y_spread"]:
        return 0.0

    # Plat punya celah antar-blok (huruf | angka | huruf); pola yang jaraknya terlalu seragam ditolak
    dx = np.diff(np.array([c[0] for c in chars], dtype=np.float32))
    block_bonus = 1.0
    if len(dx) >= 3:
        dx_cv = float(dx.std() / (dx.mean() + 1e-6))
        max_med = float(dx.max() / (np.median(dx) + 1e-6))
        if dx_cv < 0.08 and max_med < 1.15:
            return 0.0
        block_bonus = 1.25 if max_med >= 1.30 else 0.90

    char_score = min(len(chars), 8) / 8.0
    align_score = max(0.1, 1.0 - (h_cv + y_spread))
    sat_score = max(0.2, 1.0 - mean_sat / 110.0)
    return char_score * (contrast_gap / 100.0) * align_score * sat_score * block_bonus


def geometric_candidates(mask: np.ndarray, img_area: float, y_offset: int, w_full: int, h_full: int, cfg=CFG):
    """Kotak kontur yang bentuknya mirip plat (rasio, luas, extent)."""
    contours, _ = cv2.findContours(mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    ar_lo, ar_hi = cfg["plate_ar_range"]
    area_lo, area_hi = cfg["plate_area_range"]
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if h < 10 or w < 30:
            continue
        if not (ar_lo <= w / float(h) <= ar_hi):
            continue
        if not (area_lo <= (w * h) / img_area <= area_hi):
            continue
        if cv2.contourArea(cnt) / float(w * h) < cfg["plate_min_extent"]:
            continue
        if (y + y_offset + h / 2.0) / h_full > 0.95:
            continue
        px, py = int(round(0.05 * w)), int(round(0.05 * h))
        box = (max(0, x - px), max(0, y + y_offset - py), min(w_full, x + w + px), min(h_full, y + h + y_offset + py))
        boxes.append((w * h, box))
    return boxes


def iou(a, b) -> float:
    """Intersection over Union dua kotak (x1, y1, x2, y2)."""
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def nms(scored, iou_thr: float):
    """Non-maximum suppression: kotak yang tumpang tindih dengan kotak berskor lebih tinggi dibuang."""
    keep = []
    for box, score in sorted(scored, key=lambda c: c[1], reverse=True):
        if all(iou(box, k[0]) <= iou_thr for k in keep):
            keep.append((box, score))
    return keep


def localize_plates(img_bgr: np.ndarray, cfg=CFG, return_steps: bool = False):
    """Semua ROI plat [((x1, y1, x2, y2), skor), ...] dalam koordinat citra penuh, urut dari skor tertinggi."""
    h_full, w_full = img_bgr.shape[:2]
    y_offset = int(h_full * cfg["crop_top_ratio"])
    gray, enhanced, denoised = enhance(img_bgr[y_offset:, :], cfg)
    img_area = float(h_full * w_full)
    k_open = cv2.getStructuringElement(cv2.MORPH_RECT, cfg["open_kernel"])

    # Cabang 1: Sobel-X + Otsu + closing/opening (Topik 09, 07, 08)
    sobel = cv2.convertScaleAbs(cv2.Sobel(cv2.GaussianBlur(denoised, (5, 5), 0), cv2.CV_32F, 1, 0, ksize=3))
    _, sobel_bin = cv2.threshold(sobel, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k_close_s = cv2.getStructuringElement(cv2.MORPH_RECT, cfg["sobel_close_kernel"])
    morph_sobel = cv2.morphologyEx(cv2.morphologyEx(sobel_bin, cv2.MORPH_CLOSE, k_close_s), cv2.MORPH_OPEN, k_open)

    # Cabang 2: Canny + closing/opening (Topik 09, 08)
    v = float(np.median(denoised))
    canny = cv2.Canny(denoised, int(max(30, 0.66 * v)), int(min(255, 1.33 * v)))
    k_close_c = cv2.getStructuringElement(cv2.MORPH_RECT, cfg["canny_close_kernel"])
    morph_canny = cv2.morphologyEx(cv2.morphologyEx(canny, cv2.MORPH_CLOSE, k_close_c), cv2.MORPH_OPEN, k_open)

    cands = geometric_candidates(morph_sobel, img_area, y_offset, w_full, h_full, cfg) + \
        geometric_candidates(morph_canny, img_area, y_offset, w_full, h_full, cfg)
    cands.sort(key=lambda c: c[0], reverse=True)

    scored, seen = [], []
    for _, box in cands:
        if len(seen) >= cfg["max_candidates"]:
            break
        cx, cy = (box[0] + box[2]) / 2.0, (box[1] + box[3]) / 2.0
        if any(abs(cx - sx) < 10 and abs(cy - sy) < 10 for sx, sy in seen):
            continue
        seen.append((cx, cy))
        score = score_plate_candidate(img_bgr[box[1]:box[3], box[0]:box[2]], cfg)
        if score > 0:
            scored.append((box, score))

    found = [p for p in nms(scored, cfg["nms_iou"]) if p[1] >= cfg["min_plate_score"]][: cfg["max_plates"]]
    if return_steps:
        return found, {"gray": gray, "enhanced": enhanced, "denoised": denoised, "sobel_bin": sobel_bin,
                       "morph_sobel": morph_sobel, "canny": canny, "morph_canny": morph_canny,
                       "n_candidates": len(seen), "n_verified": len(scored)}
    return found

### 3.1 Evaluasi deteksi ROI per plat

Tiap ROI hasil deteksi dicocokin satu-satu sama plat GT di citra yang sama. Pasangan dengan IoU paling tinggi dicocokin duluan, dan satu plat GT cuma boleh dipasangin ke satu ROI. Kalau ROI-nya kena plat GT (IoU minimal `iou_match`), dihitung sebagai TP. Kalau ROI-nya nggak kena plat GT mana pun, misalnya malah nangkep stiker, lampu, atau teks spanduk, dihitung FP. Sedangkan plat GT yang nggak berhasil ditemuin dihitung FN.

Dari ketiganya, precision dihitung dari TP dibagi (TP + FP), recall dari TP dibagi (TP + FN), dan F1 itu rata-rata dari precision dan recall.

In [ ]:
PLATES_BY_IMAGE = {key: grp for key, grp in plates.groupby(["split", "image_name"])}


def gt_plates(split: str, image_name: str, img_shape):
    """Plat GT dalam piksel, hanya untuk pencocokan dan evaluasi."""
    H, W = img_shape[:2]
    out = []
    for _, p in PLATES_BY_IMAGE[(split, image_name)].iterrows():
        box = (int((p["cx"] - p["bw"] / 2) * W), int((p["cy"] - p["bh"] / 2) * H),
               int((p["cx"] + p["bw"] / 2) * W), int((p["cy"] + p["bh"] / 2) * H))
        digit = None if pd.isna(p["last_digit"]) else int(p["last_digit"])
        out.append({"box": box, "text": p["plate_text"], "digit": digit})
    return out


def match_plates(pred_boxes, gt_boxes, iou_thr: float):
    """Pencocokan satu-satu (greedy, IoU tertinggi dulu). Mengembalikan [(i_pred, j_gt, iou), ...]."""
    pairs = sorted(((iou(p, g), i, j) for i, p in enumerate(pred_boxes) for j, g in enumerate(gt_boxes)), reverse=True)
    used_p, used_g, matches = set(), set(), []
    for v, i, j in pairs:
        if v < iou_thr:
            break
        if i in used_p or j in used_g:
            continue
        used_p.add(i)
        used_g.add(j)
        matches.append((i, j, v))
    return matches


def process_split(split: str, cfg=CFG, transform=None, segment: bool = True, desc=None):
    """Deteksi semua ROI (dan segmentasi digit di tiap ROI) untuk setiap citra pada satu split.

    ROI disimpan dengan ambang skor 0 supaya `min_plate_score` bisa dipilih belakangan tanpa menjalankan ulang.
    """
    cfg_all = {**cfg, "min_plate_score": 0.0}
    records = []
    sub = images[images["split"] == split]
    for _, r in tqdm(sub.iterrows(), total=len(sub), desc=desc or f"Deteksi {split}", leave=False):
        img = cv2.imread(r["image_path"])
        if img is None:
            continue
        if transform is not None:
            img = transform(img)
        preds = []
        for box, score in localize_plates(img, cfg_all):
            patch = None
            if segment:
                patch, _ = segment_last_digit(img[box[1]:box[3], box[0]:box[2]], cfg)
            preds.append({"box": box, "score": score, "patch": patch})
        records.append({"image_name": r["image_name"], "image_path": r["image_path"],
                        "gts": gt_plates(split, r["image_name"], img.shape), "preds": preds})
    return records


def evaluate_detection(records, min_score: float, iou_thr: float = CFG["iou_match"]) -> dict:
    tp = fp = fn = 0
    ious = []
    for r in records:
        preds = [p for p in r["preds"] if p["score"] >= min_score]
        m = match_plates([p["box"] for p in preds], [g["box"] for g in r["gts"]], iou_thr)
        tp += len(m)
        fp += len(preds) - len(m)
        fn += len(r["gts"]) - len(m)
        ious += [v for _, _, v in m]
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"min_plate_score": min_score, "Precision": precision, "Recall": recall, "F1": f1,
            "TP": tp, "FP": fp, "FN": fn, "Mean IoU (TP)": float(np.mean(ious)) if ious else np.nan}

### 3.2 Visualisasi tahapan lokalisasi

Contohnya diambil dari citra train yang plat-nya lebih dari satu. Kotak merah itu ROI hasil deteksi (lengkap sama skornya), sedangkan kotak hijau itu plat GT yang cuma ditampilin buat pembanding.

In [ ]:
sample = images[(images["split"] == "train") & (images["n_plates"] >= 2)].iloc[0]
img = cv2.imread(sample["image_path"])
found, steps = localize_plates(img, return_steps=True)
vis = img.copy()
for g in gt_plates("train", sample["image_name"], img.shape):
    cv2.rectangle(vis, g["box"][:2], g["box"][2:], (0, 200, 0), 3)
font_scale = max(1.0, img.shape[1] / 1500)
for box, score in found:
    cv2.rectangle(vis, box[:2], box[2:], (0, 0, 255), 3)
    cv2.putText(vis, f"{score:.2f}", (box[0], max(0, box[1] - 8)), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 255), 3)

panels = [(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB),
           f"{len(found)} ROI terdeteksi | GT {sample['n_plates']} plat | kandidat dinilai {steps['n_candidates']}"),
          (steps["gray"], "1. Grayscale (bagian bawah)"), (steps["enhanced"], "2. Enhancement"),
          (steps["denoised"], "3. Median filter"), (steps["sobel_bin"], "4. Sobel-X + Otsu"),
          (steps["morph_sobel"], "5. Closing + opening (Sobel)"), (steps["canny"], "6. Canny"),
          (steps["morph_canny"], "7. Closing + opening (Canny)")]
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, (im, title) in zip(axes.ravel(), panels):
    ax.imshow(im, cmap=None if im.ndim == 3 else "gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3.3 Milih parameter deteksi di split val

Di sini kita coba beberapa kombinasi `crop_top_ratio`, metode enhancement, dan ambang skor `min_plate_score` di split **val**. Konfigurasi yang F1-nya paling tinggi yang nanti dipakai buat test. Ambang skor ini ada trade-off-nya yaitu kalau ambangnya rendah, plat yang ketemu jadi lebih banyak (recall naik), tapi ROI salahnya juga ikut banyak (precision turun).

In [ ]:
RUN_SWEEP = True
SCORE_THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.30]

if RUN_SWEEP:
    sweep_rows = []
    for ratio in [0.30, 0.40, 0.50]:
        for method in ["clahe", "he"]:
            cfg_try = {**CFG, "crop_top_ratio": ratio, "enhance_method": method}
            recs_try = process_split("val", cfg_try, segment=False, desc=f"val ratio={ratio} {method}")
            for thr in SCORE_THRESHOLDS:
                sweep_rows.append({"crop_top_ratio": ratio, "enhance_method": method, **evaluate_detection(recs_try, thr)})
    sweep_df = pd.DataFrame(sweep_rows).sort_values("F1", ascending=False).reset_index(drop=True)
    display(sweep_df.head(10).style.format({"min_plate_score": "{:.2f}", "Precision": "{:.1%}", "Recall": "{:.1%}",
                                            "F1": "{:.1%}", "Mean IoU (TP)": "{:.3f}"}))
    CFG["crop_top_ratio"] = float(sweep_df.loc[0, "crop_top_ratio"])
    CFG["enhance_method"] = str(sweep_df.loc[0, "enhance_method"])
    CFG["min_plate_score"] = float(sweep_df.loc[0, "min_plate_score"])
    sweep_df.to_csv(os.path.join(OUT_DIR, "sweep_deteksi_val.csv"), index=False)

    # Tarik-ulur precision vs recall untuk konfigurasi terpilih
    best_cfg_rows = sweep_df[(sweep_df["crop_top_ratio"] == CFG["crop_top_ratio"]) &
                             (sweep_df["enhance_method"] == CFG["enhance_method"])].sort_values("min_plate_score")
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(best_cfg_rows["min_plate_score"], best_cfg_rows["Precision"], marker="o", label="Precision")
    ax.plot(best_cfg_rows["min_plate_score"], best_cfg_rows["Recall"], marker="o", label="Recall")
    ax.plot(best_cfg_rows["min_plate_score"], best_cfg_rows["F1"], marker="o", label="F1")
    ax.set_xlabel("min_plate_score")
    ax.set_ylabel("Nilai (val)")
    ax.set_title(f"Deteksi ROI di val: crop_top_ratio={CFG['crop_top_ratio']}, {CFG['enhance_method']}")
    ax.legend()
    plt.tight_layout()
    plt.show()

print("Konfigurasi deteksi terpilih:",
      {k: CFG[k] for k in ["crop_top_ratio", "enhance_method", "min_plate_score"]})